In [ ]:
import glob, os, subprocess, sys
REPO = glob.glob("/kaggle/input/**/arcagi-arc-agi-1", recursive=True)[0]

DRY_RUN, LIMIT, CANDIDATES, GEN_BATCH, MAX_TOKENS = True, 400, 4, 8, 512

cmd = [sys.executable, "scripts/build_stage_b.py",
       "--teacher", "local", "--model", "Qwen/Qwen2.5-Coder-7B-Instruct",
       "--arc", "data/ARC/data", "--larc", "data/LARC/dataset", "--build", "build",
       "--out", "/kaggle/working/stage_b_examples.json",
       "--limit", str(LIMIT), "--candidates", str(CANDIDATES),
       "--gen-batch", str(GEN_BATCH), "--max-tokens", str(MAX_TOKENS), "--run"]
if not DRY_RUN:
    cmd.append("--run")
subprocess.run(cmd, cwd=REPO, env={**os.environ, "PYTHONPATH": REPO}, check=True)

[local teacher] forcing --workers 1 (was 4); GPU model is single-threaded
tasks processed     : 400 (of 400)
verified descriptions: 1143
generations         : 4572  (4 candidates x 1143 descriptions)
teacher backend      : local (Qwen/Qwen2.5-Coder-7B-Instruct)

RUNNING local teacher (Qwen/Qwen2.5-Coder-7B-Instruct); writing -> /kaggle/working/stage_b_examples.json


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:  56%|█████▋    | 191/339 [00:05<00:03, 43.33it/s, Materializing param=model.layers.15.self_attn.q_proj.bias]          

  25/1143 descriptions | 0 verified (0% coverage) -> /kaggle/working/stage_b_examples.json
  50/1143 descriptions | 1 verified (0% coverage) -> /kaggle/working/stage_b_examples.json
  75/1143 descriptions | 1 verified (0% coverage) -> /kaggle/working/stage_b_examples.json


In [1]:
import glob, sys
from pathlib import Path
REPO = glob.glob("/kaggle/input/**/arcagi-arc-agi-1", recursive=True)[0]
sys.path.insert(0, REPO)
from transformers import AutoTokenizer
from arcagi.data import load_arc_directory
from arcagi.larc import load_larc
from arcagi.teacher import build_teacher_prompt, TEACHER_SYSTEM

tok   = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct")
tasks = load_arc_directory(Path(REPO) / "data/ARC/data/training")
descs = load_larc(Path(REPO) / "data/LARC/dataset", verified_only=True)

lens = []
for tid, task in list(tasks.items())[:60]:
    for d in descs.get(tid, []):
        text = tok.apply_chat_template(
            [{"role": "system", "content": TEACHER_SYSTEM},
             {"role": "user",   "content": build_teacher_prompt(task, d, True)}],
            tokenize=False, add_generation_prompt=True)
        lens.append(len(tok(text)["input_ids"]))

over = sum(1 for n in lens if n > 3072)
print(f"{len(lens)} prompts | over 3072: {over} ({over/len(lens):.0%})")
print("median", sorted(lens)[len(lens)//2], " max", max(lens))

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

175 prompts | over 3072: 57 (33%)
median 2000  max 16503


In [2]:
import glob, sys
from pathlib import Path
REPO = glob.glob("/kaggle/input/**/arcagi-arc-agi-1", recursive=True)[0]
sys.path.insert(0, REPO)
from arcagi.config import Config
from arcagi.data import load_arc_directory
from arcagi.larc import load_larc
from arcagi.teacher import LocalHFTeacher
from arcagi.sandbox import PersistentSandbox, verify_against_demos

cfg   = Config()
tasks = load_arc_directory(Path(REPO) / "data/ARC/data/training")
descs = load_larc(Path(REPO) / "data/LARC/dataset", verified_only=True)
tid   = next(t for t in tasks if descs.get(t))
task, desc = tasks[tid], descs[tid][0]

teacher = LocalHFTeacher(model="Qwen/Qwen2.5-Coder-7B-Instruct", max_tokens=512)
cands = teacher.generate(task, desc, 2)
print(f"task {tid}: {len(cands)} candidates extracted from 2 generations\n")
for i, c in enumerate(cands):
    print(f"--- candidate {i} ({len(c)} chars) ---\n{c[:1200]}\n")

with PersistentSandbox(cfg.sandbox) as sb:
    print("NUMPY PROBE:", sb.run("import numpy as np\ndef transform(g): return [[1]]", [[0]]))
    print("PLAIN PROBE:", sb.run("def transform(g): return [[1]]", [[0]]))
    for i, c in enumerate(cands):
        print(f"candidate {i} raw run :", sb.run(c, task.train_pairs[0][0]))
        print(f"candidate {i} verifies:", verify_against_demos(c, task.train_pairs, sb))

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

task 007bbfb7: 2 candidates extracted from 2 generations

--- candidate 0 (209 chars) ---
import numpy as np

def transform(grid):
    input_array = np.array(grid)
    color_pixels = np.sum(input_array != 0)
    repeated_grid = np.kron(input_array, np.ones((3, 3)))
    return repeated_grid.tolist()

--- candidate 1 (626 chars) ---
def transform(grid):
    import numpy as np
    
    # Convert the grid to a numpy array for easier manipulation
    arr = np.array(grid)
    
    # Find the number of non-zero elements (colored pixels)
    num_pixels = np.sum(arr > 0)
    
    # Repeat the array 'num_pixels' times along both axes to create the transformed grid
    transformed_arr = np.tile(arr, (num_pixels, num_pixels))
    
    # Ensure the transformed array has the correct shape (9x9)
    if transformed_arr.shape != (9, 9):
        raise ValueError("The transformed grid does not have the expected 9x9 shape.")
    
    return transformed_arr.tolist()

NUMPY PROBE: ExecResult(status=<ExecSta

In [ ]:
# ── Stage B teacher CALIBRATION: truncation fixed + 16 candidates ──────────
import glob, json, sys, time
from pathlib import Path

REPO = glob.glob("/kaggle/input/**/arcagi-arc-agi-1", recursive=True)[0]
sys.path.insert(0, REPO)

from arcagi.config import Config
from arcagi.data import load_arc_directory, build_stage_b_examples
from arcagi.larc import load_larc
from arcagi.sandbox import PersistentSandbox
from arcagi.teacher import (LocalHFTeacher, TEACHER_SYSTEM, build_teacher_prompt,
                            extract_code, distill)
from arcagi.vocab import SharedVocab

LIMIT      = 20     # same 20 tasks as the 1/75 baseline — keep fixed so it compares
CANDIDATES = 16     # was 4. Verification is free; this is the main lever.
GEN_BATCH  = 8      # now actually batches (min(8,16)=8 per GPU call)
MAX_TOKENS = 768    # was 512; headroom so code isn't cut before its closing fence
MAX_PROMPT = 8192   # was 3072, silently decapitating 33% of prompts
MODEL      = "Qwen/Qwen2.5-Coder-7B-Instruct"
OUT        = Path("/kaggle/working/stage_b_calib.json")


class FixedTeacher(LocalHFTeacher):
    """LocalHFTeacher with the prompt truncation corrected.

    The original trims the RIGHT side of the chat-templated string, cutting off the
    trailing '<|im_start|>assistant' header that add_generation_prompt appends. The
    model then continues the user turn instead of answering, emits no code fence,
    and every candidate is silently dropped. This keeps the header and trims only
    the demo-grid overflow inside the user message.
    """

    def generate(self, task, description, n):
        msgs = [{"role": "system", "content": TEACHER_SYSTEM},
                {"role": "user",
                 "content": build_teacher_prompt(task, description, self.include_grids)}]
        body   = self._tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        full   = self._tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        header = full[len(body):]

        body_ids   = self._tok(body, return_tensors="pt")["input_ids"]
        header_ids = self._tok(header, return_tensors="pt",
                               add_special_tokens=False)["input_ids"]

        budget = self.max_prompt_tokens - header_ids.shape[1]
        if body_ids.shape[1] > budget:
            body_ids = body_ids[:, :budget]
        ids = self._torch.cat([body_ids, header_ids], dim=1).to(self._input_device)
        inputs = {"input_ids": ids, "attention_mask": self._torch.ones_like(ids)}
        prompt_len = ids.shape[1]

        out, remaining = [], n
        while remaining > 0:
            b = min(self.gen_batch, remaining)
            for completion in self._generate_batch(inputs, b, prompt_len):
                if code := extract_code(completion):
                    out.append(code)
            remaining -= b
            self._empty_cache()
        return out


cfg   = Config()
vocab = SharedVocab.load(Path(REPO) / "build" / "vocab.json")
tasks = load_arc_directory(Path(REPO) / "data/ARC/data/training")
descs = load_larc(Path(REPO) / "data/LARC/dataset", verified_only=True)

teacher = FixedTeacher(model=MODEL, max_tokens=MAX_TOKENS,
                       max_prompt_tokens=MAX_PROMPT, gen_batch=GEN_BATCH)

t0, seen = time.time(), 0
def on_progress(v):
    global seen
    seen += 1
    if seen % 5 == 0 or seen == v.n_descriptions:
        el = time.time() - t0
        print(f"  {seen}/{v.n_descriptions} desc | {v.n_verified} verified "
              f"| {v.n_candidates} cands | {el/60:.1f} min ({el/seen:.0f}s/desc)", flush=True)

with PersistentSandbox(cfg.sandbox) as sandbox:
    verified = distill(tasks, descs, teacher, sandbox, n_candidates=CANDIDATES,
                       limit=LIMIT, max_workers=1, on_progress=on_progress)

examples = build_stage_b_examples(verified.pairs, vocab)
OUT.write_text(json.dumps([{"prompt": e.prompt_ids, "target": e.target_ids,
                            "task_id": e.task_id} for e in examples]))

el   = time.time() - t0
rate = verified.n_verified / max(1, verified.n_descriptions)
print("\n" + "=" * 62)
print(f"descriptions     : {verified.n_descriptions}")
print(f"verified         : {verified.n_verified}   ({rate:.1%} of descriptions)")
print(f"candidates run   : {verified.n_candidates}")
print(f"tasks covered    : {len({t for t, _, _ in verified.pairs})} / {LIMIT}")
print(f"examples written : {len(examples)} -> {OUT}")
print(f"elapsed          : {el/60:.1f} min   ({el/max(1,seen):.0f}s per description)")
print(f"\nBASELINE to beat : 1 verified / 75 descriptions  (1.3%)")
print(f"projected 400-task pass at this rate: {el/max(1,seen)*1143/3600:.1f} h")

In [1]:
# ── Stage B teacher CALIBRATION: truncation fixed + 16 candidates ──────────
import glob, json, sys, time
from pathlib import Path

REPO = glob.glob("/kaggle/input/**/arcagi-arc-agi-1", recursive=True)[0]
sys.path.insert(0, REPO)

from arcagi.config import Config
from arcagi.data import load_arc_directory, build_stage_b_examples
from arcagi.larc import load_larc
from arcagi.sandbox import PersistentSandbox
from arcagi.teacher import (LocalHFTeacher, TEACHER_SYSTEM, build_teacher_prompt,
                            extract_code, distill)
from arcagi.vocab import SharedVocab

LIMIT      = 20     # same 20 tasks as the 1/75 baseline — keep fixed so it compares
CANDIDATES = 16     # was 4. Verification is free; this is the main lever.
GEN_BATCH  = 8      # now actually batches (min(8,16)=8 per GPU call)
MAX_TOKENS = 768    # was 512; headroom so code isn't cut before its closing fence
MAX_PROMPT = 8192   # was 3072, silently decapitating 33% of prompts
MODEL      = "Qwen/Qwen2.5-Coder-7B-Instruct"
OUT        = Path("/kaggle/working/stage_b_calib.json")


class FixedTeacher(LocalHFTeacher):
    """LocalHFTeacher with the prompt truncation corrected.

    The original trims the RIGHT side of the chat-templated string, cutting off the
    trailing '<|im_start|>assistant' header that add_generation_prompt appends. The
    model then continues the user turn instead of answering, emits no code fence,
    and every candidate is silently dropped. This keeps the header and trims only
    the demo-grid overflow inside the user message.
    """

    def generate(self, task, description, n):
        msgs = [{"role": "system", "content": TEACHER_SYSTEM},
                {"role": "user",
                 "content": build_teacher_prompt(task, description, self.include_grids)}]
        body   = self._tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        full   = self._tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        header = full[len(body):]

        body_ids   = self._tok(body, return_tensors="pt")["input_ids"]
        header_ids = self._tok(header, return_tensors="pt",
                               add_special_tokens=False)["input_ids"]

        budget = self.max_prompt_tokens - header_ids.shape[1]
        if body_ids.shape[1] > budget:
            body_ids = body_ids[:, :budget]
        ids = self._torch.cat([body_ids, header_ids], dim=1).to(self._input_device)
        inputs = {"input_ids": ids, "attention_mask": self._torch.ones_like(ids)}
        prompt_len = ids.shape[1]

        out, remaining = [], n
        while remaining > 0:
            b = min(self.gen_batch, remaining)
            for completion in self._generate_batch(inputs, b, prompt_len):
                if code := extract_code(completion):
                    out.append(code)
            remaining -= b
            self._empty_cache()
        return out


cfg   = Config()
vocab = SharedVocab.load(Path(REPO) / "build" / "vocab.json")
tasks = load_arc_directory(Path(REPO) / "data/ARC/data/training")
descs = load_larc(Path(REPO) / "data/LARC/dataset", verified_only=True)

teacher = FixedTeacher(model=MODEL, max_tokens=MAX_TOKENS,
                       max_prompt_tokens=MAX_PROMPT, gen_batch=GEN_BATCH)

t0, seen = time.time(), 0
def on_progress(v):
    global seen
    seen += 1
    if seen % 5 == 0 or seen == v.n_descriptions:
        el = time.time() - t0
        print(f"  {seen}/{v.n_descriptions} desc | {v.n_verified} verified "
              f"| {v.n_candidates} cands | {el/60:.1f} min ({el/seen:.0f}s/desc)", flush=True)

with PersistentSandbox(cfg.sandbox) as sandbox:
    verified = distill(tasks, descs, teacher, sandbox, n_candidates=CANDIDATES,
                       limit=LIMIT, max_workers=1, on_progress=on_progress)

examples = build_stage_b_examples(verified.pairs, vocab)
OUT.write_text(json.dumps([{"prompt": e.prompt_ids, "target": e.target_ids,
                            "task_id": e.task_id} for e in examples]))

el   = time.time() - t0
rate = verified.n_verified / max(1, verified.n_descriptions)
print("\n" + "=" * 62)
print(f"descriptions     : {verified.n_descriptions}")
print(f"verified         : {verified.n_verified}   ({rate:.1%} of descriptions)")
print(f"candidates run   : {verified.n_candidates}")
print(f"tasks covered    : {len({t for t, _, _ in verified.pairs})} / {LIMIT}")
print(f"examples written : {len(examples)} -> {OUT}")
print(f"elapsed          : {el/60:.1f} min   ({el/max(1,seen):.0f}s per description)")
print(f"\nBASELINE to beat : 1 verified / 75 descriptions  (1.3%)")
print(f"projected 400-task pass at this rate: {el/max(1,seen)*1143/3600:.1f} h")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  5/53 desc | 1 verified | 80 cands | 28.4 min (341s/desc)
  10/53 desc | 1 verified | 160 cands | 49.5 min (297s/desc)


OutOfMemoryError: CUDA out of memory. Tried to allocate 7.00 GiB. GPU 0 has a total capacity of 14.56 GiB of which 4.40 GiB is free. Including non-PyTorch memory, this process has 10.16 GiB memory in use. Of the allocated memory 9.82 GiB is allocated by PyTorch, and 214.61 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)